# Executed small-scale adaptation: PowerMLP

**Original paper:** Ruichen Qiu, Yibo Miao, Shiwen Wang, and Yifan Zhu, *PowerMLP: An Efficient Version of KAN*, AAAI 39(19), 20069–20076 (2025). DOI: https://doi.org/10.1609/aaai.v39i19.34210.

This notebook implements the paper’s published layer formula for a small regression task: for every non-final layer, `z = α b(x) + ReLU_k(Wx + γ)`, where `b(x)=x/(1+e^-x)` and `ReLU_k(u)=max(0,u)^k`; the final layer is affine. The NumPy implementation is a small adaptation, not the paper’s benchmark suite or a claim about its reported speed.

In [1]:
import json, time
from pathlib import Path
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

SEED=42; rng=np.random.default_rng(SEED)
x=np.linspace(-3,3,600).reshape(-1,1); y=(np.sin(2*x[:,0])+0.35*x[:,0]**2-0.15*x[:,0]**3).astype(float)
perm=rng.permutation(len(x)); tr=perm[:450]; te=perm[450:]; Xtr,Xte=x[tr],x[te]; ytr,yte=y[tr],y[te]

def basis(x): return x/(1.0+np.exp(-x))
def relu_k(x,k=2): return np.maximum(x,0.0)**k

class PowerMLP:
    # Exact adapted layer: alpha*b(x) + ReLU_k(Wx + gamma), then final affine.
    def __init__(self, widths=(16,16), k=2, lr=0.01, epochs=1800):
        self.widths=widths; self.k=k; self.lr=lr; self.epochs=epochs
    def fit(self,X,y):
        rng=np.random.default_rng(SEED); dims=[X.shape[1],*self.widths,1]
        self.W=[rng.normal(0,.35,(dims[i+1],dims[i])) for i in range(len(dims)-1)]
        self.b=[np.zeros(dims[i+1]) for i in range(len(dims)-1)]
        self.A=[rng.normal(0,.05,(dims[i+1],dims[i])) for i in range(len(dims)-2)]
        n=len(X)
        for _ in range(self.epochs):
            xs=[X]; zs=[]; hs=[]
            for i in range(len(self.W)-1):
                z=xs[-1]@self.W[i].T+self.b[i]; h=relu_k(z,self.k)+xs[-1]@self.A[i].T; zs.append(z); hs.append(h); xs.append(h)
            out=xs[-1]@self.W[-1].T+self.b[-1]; err=(out[:,0]-y); d=2*err[:,None]/n
            gW=[None]*len(self.W); gb=[None]*len(self.b); gA=[None]*len(self.A)
            gW[-1]=d.T@xs[-1]; gb[-1]=d.sum(0); dx=d@self.W[-1]
            for i in reversed(range(len(self.W)-1)):
                xprev=xs[i]; dz=dx*(self.k*np.maximum(zs[i],0)**(self.k-1)*(zs[i]>0)); gW[i]=dz.T@xprev; gb[i]=dz.sum(0); gA[i]=dx.T@basis(xprev); dx=dz@self.W[i]+dx@self.A[i]
            for i in range(len(self.W)): self.W[i]-=self.lr*gW[i]; self.b[i]-=self.lr*gb[i]
            for i in range(len(self.A)): self.A[i]-=self.lr*gA[i]
        return self
    def predict(self,X):
        h=X
        for i in range(len(self.W)-1): h=relu_k(h@self.W[i].T+self.b[i],self.k)+h@self.A[i].T
        return (h@self.W[-1].T+self.b[-1])[:,0]

def evaluate(name, estimator):
    t=time.perf_counter(); estimator.fit(Xtr,ytr); elapsed=time.perf_counter()-t; pred=estimator.predict(Xte)
    return {"condition":name,"rmse":round(float(mean_squared_error(yte,pred)**0.5),6),"mae":round(float(mean_absolute_error(yte,pred)),6),"fit_seconds":round(float(elapsed),6),"n_train":len(Xtr),"n_test":len(Xte)}

results=[evaluate("one-hidden-layer ReLU MLP",MLPRegressor(hidden_layer_sizes=(16,16),solver="lbfgs",random_state=SEED,max_iter=500)),evaluate("published-formula PowerMLP adaptation",PowerMLP())]
out={"task":"deterministic one-dimensional regression","formula":"z = alpha*b(x) + ReLU_k(Wx + gamma), b(x)=x/(1+exp(-x)), k=2; final layer affine","seed":SEED,"results":results,"limitation":"small NumPy adaptation, not the authors' benchmark or speed claim"}
Path("results.json").write_text(json.dumps(out,indent=2)); print(json.dumps(out,indent=2))

{
  "task": "deterministic one-dimensional regression",
  "formula": "z = alpha*b(x) + ReLU_k(Wx + gamma), b(x)=x/(1+exp(-x)), k=2; final layer affine",
  "seed": 42,
  "results": [
    {
      "condition": "one-hidden-layer ReLU MLP",
      "rmse": 0.113506,
      "mae": 0.065012,
      "fit_seconds": 0.203723,
      "n_train": 450,
      "n_test": 150
    },
    {
      "condition": "published-formula PowerMLP adaptation",
      "rmse": 0.163741,
      "mae": 0.117244,
      "fit_seconds": 2.5377,
      "n_train": 450,
      "n_test": 150
    }
  ],
  "limitation": "small NumPy adaptation, not the authors' benchmark or speed claim"
}
